# Reproducible topology-aware BWB optimization

This notebook is the readable orchestration layer for the public pipeline.
Implementation details live in `src/bwb_pipeline/`; the important model,
constraint, budget, optimization, and reporting calls remain visible here.

Scientific contract:

- one public `MASTER_SEED`, with stable named child streams recorded in the manifest;
- forward model: exactly 21 geometry/structure inputs → weight, payload, fuel;
- stress classifier: all 24 inputs, class 1 means stress ≤ 335 MPa;
- nominal calibrated probability ≥ 0.90;
- median probability ≥ 0.80 at both configured perturbation levels;
- any L/D warning, error, non-finite result, or non-positive CD is a hard rejection;
- three independently seeded CMA-ES trajectories, multi-round warm/cold search, then projected-gradient proposals;
- one held-out confirmation bank; no confirmation feedback into search;
- exactly one final design per mission case.

The reported Pareto set is an **empirical nondominated archive**, not a proof of
the complete Pareto front or global optimum.


## 0. Start the kernel reproducibly

`PYTHONHASHSEED` must be set before Python starts. Launch Jupyter from the
repository root with the same process-level variables used by
`scripts/run_reproducible.py`. The code below sets safe defaults for libraries
that have not yet initialized, but it cannot retroactively change Python's hash
seed.


In [1]:
import os
from pathlib import Path
import sys

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "bwb_pipeline").is_dir():
    raise RuntimeError("Start this notebook from the repository root or notebooks/.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)
print("PYTHONHASHSEED at process start:", os.environ.get("PYTHONHASHSEED"))


Project root: /Users/hmu2718/Downloads/bwb_reproducible_pipeline_2r_0.5
PYTHONHASHSEED at process start: None


## 1. Load configuration and create all named random streams

`MASTER_SEED` is the only user-controlled random input. SHA-256 namespaces
derive independent streams, so the three CMA repeats are reproducible without
being artificially correlated.


In [ ]:
import json
import numpy as np
import pandas as pd
import yaml

from bwb_pipeline.config import pipeline_config_from_mapping
from bwb_pipeline.reproducibility import SeedRegistry, set_global_determinism

CONFIG_PATH = PROJECT_ROOT / "configs" / "ensemble5_noise0p5.yaml"
raw_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config = pipeline_config_from_mapping(raw_config)
MASTER_SEED = config.master_seed
seedbook = SeedRegistry(MASTER_SEED)
determinism_report = set_global_determinism(
    seedbook.derive("pipeline/global", upper_bound=2**32),
    strict=config.determinism.strict,
    torch_num_threads=config.determinism.torch_num_threads,
)
display(pd.Series(determinism_report, name="value"))


seed                               2314182082
strict                                   True
pythonhashseed_at_process_start          None
cublas_workspace_config               :4096:8
torch_available                          True
torch_version                          2.13.0
cuda_available                          False
torch_num_threads                           1
Name: value, dtype: object

## 2. Authoritative schema and leakage-resistant 70/15/15 split

The split unit is the exact 21D physical design, not a CSV row. Therefore the
same geometry/structure observed under multiple flying conditions cannot leak
across train, validation, and test. Forward targets are checked for invariance
within each 21D group before a flight-free model is allowed.


In [3]:
from bwb_pipeline.data import prepare_data
from bwb_pipeline.manifest import hash_existing_files, hash_mapping
from bwb_pipeline.pipeline import guard_run_identity, project_source_files
from bwb_pipeline.schema import (
    ALL_INPUT_COLUMNS,
    ALL_TARGET_COLUMNS,
    CONTINUOUS_DESIGN_COLUMNS,
    DESIGN_COLUMNS,
    FLIGHT_COLUMNS,
    FORWARD_TARGET_COLUMNS,
    STRESS_TARGET_COLUMN,
    TOPOLOGY_COLUMNS,
)

OUTPUT_ROOT = PROJECT_ROOT / config.project.output_root
OPTIMIZATION_ROOT = PROJECT_ROOT / raw_config["optimization"]["output_dir"]
SOURCE_FILES = project_source_files(PROJECT_ROOT)
source_hashes = hash_existing_files(SOURCE_FILES, base_directory=PROJECT_ROOT)
run_identity = {
    "schema": "bwb-run-identity-v2",
    "master_seed": MASTER_SEED,
    "config_sha256": hash_mapping(raw_config),
    "source_tree_sha256": hash_mapping(source_hashes),
}
guard_run_identity(OUTPUT_ROOT, run_identity)
guard_run_identity(OPTIMIZATION_ROOT, run_identity)
bundle = prepare_data(
    PROJECT_ROOT / config.project.data_path,
    config.data,
    seedbook,
    split_manifest_path=OUTPUT_ROOT / config.data.split_manifest,
)
BASE_INPUT_FILES = [CONFIG_PATH, bundle.source_path, *SOURCE_FILES]
BASE_INPUT_SNAPSHOT = hash_existing_files(
    BASE_INPUT_FILES, base_directory=PROJECT_ROOT
)

assert len(DESIGN_COLUMNS) == 21
assert len(ALL_INPUT_COLUMNS) == 24
assert set(FLIGHT_COLUMNS).isdisjoint(DESIGN_COLUMNS)

split_report = pd.DataFrame({
    "unique_designs": bundle.split_manifest.group_counts,
    "rows": bundle.split_manifest.row_counts,
    "feasible_row_fraction": bundle.split_manifest.feasible_row_fractions,
})
display(split_report)
print("Dataset fingerprint:", bundle.dataset_fingerprint)
print("Split fingerprint:  ", bundle.split_manifest.split_fingerprint)

# Preflight the user-requested topology support rule before expensive training.
from bwb_pipeline.optimization import OptimizationConfig
from bwb_pipeline.topology_budget import build_empirical_topology_prior

missions = [dict(case) for case in raw_config["test_cases"]]
opt_values = raw_config["optimization"]
opt_config = OptimizationConfig.from_mapping(opt_values)
assert opt_values["nominal_probability_threshold"] == config.stress_classifier.probability_threshold
assert opt_values["robust_probability_threshold"] == config.stress_classifier.noise_median_probability_threshold
topology_prior = build_empirical_topology_prior(
    bundle.rows,
    missions,
    stress_limit_mpa=config.stress_classifier.stress_limit_mpa,
    minimum_feasible_unique_designs=opt_values[
        "minimum_feasible_unique_designs_per_topology"
    ],
    split=opt_values["topology_prior_split"],
    eligibility_split=opt_values.get("topology_eligibility_split"),
)
topology_prior.to_csv(OUTPUT_ROOT / "topology_empirical_prior.csv", index=False)
display(topology_prior.groupby("case_id").size().rename("eligible_topologies"))


,unique_designs,rows,feasible_row_fraction
test,2055,2055,0.591241
train,9592,9595,0.591141
validation,2055,2055,0.591241


Dataset fingerprint: accf08a215181a64894e3a6ee007cd846cd0c1cfc61abae6512bfe1eef8853c7
Split fingerprint:   e163bcc177da6743c5162c5569ce52ed79b7ece4ae01ce9ff873d9646bf7ec52


case_id
1    50
2    50
3    50
Name: eligible_topologies, dtype: int64

## 3. Train or load the 21→3 differentiable forward model

The residual MLP receives **only** `DESIGN_COLUMNS`. It uses AdamW, a validation
plateau scheduler, early stopping, and train-only input/target scaling. The
defaults are 2000 epochs, patience 300, LR patience 120, learning rate 1e-3,
and logging every 25 epochs. If an exact-compatible artifact exists it is
loaded; an incompatible old 24-input checkpoint is rejected.


In [4]:
from bwb_pipeline.models import ResidualForwardNet, train_or_load_forward

forward = train_or_load_forward(
    bundle,
    config.forward_model,
    seedbook,
    artifact_dir=PROJECT_ROOT / config.forward_model.output_dir,
    device=config.determinism.device,
)
assert tuple(forward.feature_names) == tuple(DESIGN_COLUMNS)

forward_metrics_path = PROJECT_ROOT / config.forward_model.output_dir / "metrics.json"
forward_metrics = json.loads(forward_metrics_path.read_text(encoding="utf-8"))
display(pd.DataFrame(forward_metrics["rows"]))
print("Forward artifact:", forward.artifact_id)


Loaded compatible forward artifact 54a87f93b80d8898.


,MAE,MAPE_percent,NRMSE_q05_q95,R2,RMSE,split,target
0,6.141554e-01,0.601400,0.001935,0.999968,8.779099e-01,train,Aircraft Empty Weight
1,1.247675e+06,0.247715,0.001457,0.999978,1.613022e+06,train,Payload Volume
2,6.489101e+05,0.324168,0.001856,0.999965,8.394441e+05,train,Fuel Volume
3,1.857532e+00,1.044122,0.015677,0.997807,7.307662e+00,validation,Aircraft Empty Weight
4,2.147259e+06,0.420203,0.003303,0.999881,3.766043e+06,validation,Payload Volume
5,1.161120e+06,0.561258,0.005472,0.999695,2.491220e+06,validation,Fuel Volume
6,1.540666e+00,1.008969,0.009809,0.999184,3.995958e+00,test,Aircraft Empty Weight
7,2.074154e+06,0.423326,0.003301,0.999885,3.713461e+06,test,Payload Volume
8,1.092444e+06,0.555058,0.004147,0.999824,1.896343e+06,test,Fuel Volume


Forward artifact: 54a87f93b80d8898


## 4. Train or load the 24-input calibrated stress classifier

A deterministic CPU Ordered CatBoost model is selected by group-aware CV
inside the training partition. Validation logits fit a two-parameter Platt
calibrator. The untouched test report includes log loss, ROC-AUC, PR-AUC,
Brier/ECE, and threshold diagnostics at 0.5, 0.8, and 0.9. Optimization always
uses calibrated `P(stress ≤ 335 MPa)` and keeps the 0.90 threshold fixed.


In [5]:
from bwb_pipeline.models import train_or_load_stress_classifier

stress = train_or_load_stress_classifier(
    bundle,
    config.stress_classifier,
    seedbook,
    artifact_dir=PROJECT_ROOT / config.stress_classifier.output_dir,
)
assert tuple(stress.feature_names) == tuple(ALL_INPUT_COLUMNS)

stress_metrics_path = PROJECT_ROOT / config.stress_classifier.output_dir / "metrics.json"
stress_metrics = json.loads(stress_metrics_path.read_text(encoding="utf-8"))
display(pd.json_normalize(stress_metrics["splits"], sep="."))
display(pd.DataFrame(stress_metrics["splits"]["test"]["thresholds"]).T)
print("Stress artifact:", stress.artifact_id)

MODEL_DIRECTORIES = (
    PROJECT_ROOT / config.forward_model.output_dir,
    PROJECT_ROOT / config.stress_classifier.output_dir,
)
MODEL_FILES = [
    path for directory in MODEL_DIRECTORIES
    for path in directory.rglob("*") if path.is_file()
]
MODEL_INPUT_SNAPSHOT = hash_existing_files(
    MODEL_FILES, base_directory=PROJECT_ROOT
)


Loaded compatible stress artifact f809c0b49a7b98f9.


,test.members,test.probability.average_precision_feasible,test.probability.brier_score,test.probability.ece_15_bins,test.probability.log_loss,test.probability.roc_auc,test.thresholds.0.5.accuracy,test.thresholds.0.5.balanced_accuracy,test.thresholds.0.5.false_feasible,test.thresholds.0.5.false_feasible_rate_among_true_infeasible,...,validation.thresholds.0.9.balanced_accuracy,validation.thresholds.0.9.false_feasible,validation.thresholds.0.9.false_feasible_rate_among_true_infeasible,validation.thresholds.0.9.false_infeasible,validation.thresholds.0.9.feasible_precision,validation.thresholds.0.9.feasible_recall,validation.thresholds.0.9.specificity,validation.thresholds.0.9.threshold,validation.thresholds.0.9.true_feasible,validation.thresholds.0.9.true_infeasible_predicted_infeasible
0,[{'average_precision_feasible': 0.955405633946...,0.955589,0.091674,0.027322,0.295981,0.942934,0.869586,0.864175,139,0.165476,...,0.766057,19,0.022619,541,0.972583,0.554733,0.977381,0.9,674,821


,accuracy,balanced_accuracy,false_feasible,false_feasible_rate_among_true_infeasible,false_infeasible,feasible_precision,feasible_recall,specificity,threshold,true_feasible,true_infeasible_predicted_infeasible
0.5,0.869586,0.864175,139.0,0.165476,129.0,0.886531,0.893827,0.834524,0.5,1086.0,701.0
0.8,0.812652,0.832378,50.0,0.059524,335.0,0.946237,0.724280,0.940476,0.8,880.0,790.0
0.9,0.754745,0.787816,26.0,0.030952,478.0,0.965924,0.606584,0.969048,0.9,737.0,814.0


Stress artifact: f809c0b49a7b98f9


## 5. Load the official L/D model and verify its domain flag

The adapter preserves `LD`, `CL`, `CD`, and every warning. It fails closed on
exceptions, missing keys, non-finite values, `CD <= 0`, or any warning. This is
different from merely clipping the 21 design variables to their official
bounds: the L/D surrogate has its own validity domain.


In [6]:
from bwb_pipeline.ld_adapter import REQUIRED_LD_FILES, make_ld_adapter

ld_values = raw_config["ld_model"]
if ld_values.get("reject_any_warning") is not True:
    raise ValueError("Published optimization requires reject_any_warning=true.")
if ld_values.get("require_finite_ld_cl_cd") is not True:
    raise ValueError("Published optimization requires require_finite_ld_cl_cd=true.")
LD_SOURCE_FILES = [
    PROJECT_ROOT / config.project.ld_model_dir / name
    for name in REQUIRED_LD_FILES
]
LD_INPUT_SNAPSHOT = hash_existing_files(
    LD_SOURCE_FILES, base_directory=PROJECT_ROOT
)
ld_adapter = make_ld_adapter(
    PROJECT_ROOT / config.project.ld_model_dir,
    cache=True,
    max_cache_entries=ld_values.get("cache_max_entries", 50000),
)
probe = bundle.structural_designs.loc[:, DESIGN_COLUMNS].head(1)
ld_probe = ld_adapter.predict_many(probe, missions[0])
display(ld_probe)
print("Probe accepted by L/D domain gate:", bool(ld_probe.iloc[0]["ld_in_domain"]))


,LD,CL,CD,warnings,ld_warnings,ld_warning_count,ld_evaluation_error,ld_rejection_reasons,ld_in_domain
0,8.224907,0.080092,0.009738,[],[],0,None,[],True


Probe accepted by L/D domain gate: True


## 6. Rank supported topologies and allocate a gamma-shaped budget

Eligibility follows the predeclared all-dataset support rule (>20 unique
stress-feasible designs), so it is explicitly transductive. Rank order uses
TRAIN W/P/F labels only and gives L/D full credit (zero shortfall). A monotone
gamma-rank kernel concentrates budget at the head; a uniform floor preserves
exploration, and largest-remainder allocation conserves the exact integer
evaluation budget.


In [7]:
from bwb_pipeline.topology_budget import (
    allocate_generation_blocks,
    build_round_weights,
)

display(topology_prior.groupby("case_id", sort=True).head(10))

first_case_id = int(missions[0]["case_id"])
case_one_prior = topology_prior.loc[topology_prior["case_id"] == first_case_id]
round_zero_weights = build_round_weights(
    case_one_prior,
    prior_weight=opt_config.rounds[0].prior_weight,
    gamma_shape=opt_config.gamma_shape,
    gamma_scale_fraction=opt_config.gamma_scale_fraction,
    uniform_exploration_fraction=opt_config.uniform_exploration_fraction,
)
round_zero_allocation = allocate_generation_blocks(
    round_zero_weights,
    total_evaluations=opt_config.rounds[0].total_evaluations_per_case,
    repeats=opt_config.repeats,
    population=opt_config.cma_population,
    minimum_generations_per_repeat=opt_config.minimum_generations_per_topology_repeat,
)
display(round_zero_allocation.head(15))
assert round_zero_allocation["allocated_evaluations"].sum() == opt_config.rounds[0].total_evaluations_per_case


,case_id,prior_rank,# of Ribs,# of Fuselage Ribs,# of Fuselage Spars,ranking_feasible_unique_designs,median_label_score_ld_full,q25_label_score_ld_full,q75_label_score_ld_full,feasible_unique_designs,eligibility_split,ranking_split,topology,topology_label
0,1,0,4,3,3,54,0.573690,0.514108,0.703077,70,all,train,"(4, 3, 3)",433
1,1,1,7,3,3,457,0.588947,0.527627,0.704950,670,all,train,"(7, 3, 3)",733
2,1,2,5,3,3,621,0.589895,0.521313,0.709784,854,all,train,"(5, 3, 3)",533
3,1,3,6,3,3,649,0.595381,0.524988,0.719134,906,all,train,"(6, 3, 3)",633
4,1,4,7,5,4,16,0.612339,0.575882,0.729525,21,all,train,"(7, 5, 4)",754
5,1,5,5,3,4,33,0.620404,0.526806,0.841061,49,all,train,"(5, 3, 4)",534
6,1,6,8,3,3,165,0.632103,0.564864,0.786268,236,all,train,"(8, 3, 3)",833
7,1,7,8,5,4,16,0.644546,0.620465,0.711350,28,all,train,"(8, 5, 4)",854
8,1,8,7,3,4,66,0.646310,0.553268,0.766804,86,all,train,"(7, 3, 4)",734
9,1,9,7,5,3,55,0.648140,0.578270,0.779193,83,all,train,"(7, 5, 3)",753


,case_id,prior_rank,# of Ribs,# of Fuselage Ribs,# of Fuselage Spars,ranking_feasible_unique_designs,median_label_score_ld_full,q25_label_score_ld_full,q75_label_score_ld_full,feasible_unique_designs,...,topology,topology_label,evidence_rank,prior_gamma_weight,evidence_gamma_weight,allocation_weight,generations_per_repeat,evaluations_per_repeat,allocated_evaluations,allocated_fraction
0,1,0,4,3,3,54,0.573690,0.514108,0.703077,70,...,"(4, 3, 3)",433,0,0.426634,0.426634,0.365639,1655,52960,158880,0.3310
1,1,1,7,3,3,457,0.588947,0.527627,0.704950,670,...,"(7, 3, 3)",733,1,0.191832,0.191832,0.166057,757,24224,72672,0.1514
2,1,2,5,3,3,621,0.589895,0.521313,0.709784,854,...,"(5, 3, 3)",533,2,0.115724,0.115724,0.101365,466,14912,44736,0.0932
3,1,3,6,3,3,649,0.595381,0.524988,0.719134,906,...,"(6, 3, 3)",633,3,0.076170,0.076170,0.067745,315,10080,30240,0.0630
4,1,4,7,5,4,16,0.612339,0.575882,0.729525,21,...,"(7, 5, 4)",754,4,0.052317,0.052317,0.047469,224,7168,21504,0.0448
5,1,5,5,3,4,33,0.620404,0.526806,0.841061,49,...,"(5, 3, 4)",534,5,0.036855,0.036855,0.034326,164,5248,15744,0.0328
6,1,6,8,3,3,165,0.632103,0.564864,0.786268,236,...,"(8, 3, 3)",833,6,0.026402,0.026402,0.025442,124,3968,11904,0.0248
7,1,7,8,5,4,16,0.644546,0.620465,0.711350,28,...,"(8, 5, 4)",854,7,0.019142,0.019142,0.019271,97,3104,9312,0.0194
8,1,8,7,3,4,66,0.646310,0.553268,0.766804,86,...,"(7, 3, 4)",734,8,0.014004,0.014004,0.014903,77,2464,7392,0.0154
9,1,9,7,5,3,55,0.648140,0.578270,0.779193,83,...,"(7, 5, 3)",753,9,0.010316,0.010316,0.011769,63,2016,6048,0.0126


## 7. Build the forward model

All optimizers call this evaluator. Only the 18 continuous variables receive
antithetic Sobol perturbations; the three topology counts and mission remain
fixed. Every design sees the same search bank. A separately seeded held-out
bank is loaded only after all rounds finish.


In [8]:
from bwb_pipeline.evaluator import (
    BWBEvaluator,
    EvaluationThresholds,
    make_antithetic_sobol_noise,
)
from bwb_pipeline.schema import CONTINUOUS_DESIGN_COLUMNS

search_bank = make_antithetic_sobol_noise(
    opt_values["search_noise_samples"],
    len(CONTINUOUS_DESIGN_COLUMNS),
    seedbook.derive("probability_noise/search"),
)
confirmation_bank = make_antithetic_sobol_noise(
    opt_values["confirmation_noise_samples"],
    len(CONTINUOUS_DESIGN_COLUMNS),
    seedbook.derive("probability_noise/confirmation"),
)
evaluator = BWBEvaluator(
    forward,
    stress,
    ld_adapter,
    search_noise_bank=search_bank,
    confirmation_noise_bank=confirmation_bank,
    noise_fractions=opt_values["noise_fractions_of_bound_width"],
    thresholds=EvaluationThresholds(
        nominal_probability=opt_values["nominal_probability_threshold"],
        robust_probability=opt_values["robust_probability_threshold"],
    ),
)
display(evaluator.evaluate(probe, missions[0], bank_name="search"))


,C2/C1,C3/C1,C4/C1,B1/C1,B2/C1,B3/C1,X3/C1,S1,S3,C1,...,objective_fuel_shortfall,official_loss,nominal_probability_violation,robust_probability_violation,constraint_tier,active_constraint_violation,hard_accepted,search_merit,robustness_bank,robustness_samples
0,0.737265,0.260749,0.085564,0.134909,0.115561,0.548163,0.529724,40.234322,23.89542,3798.992823,...,0.146622,1.710772,0.0,0.0,0,0.0,True,0.631101,search,64


## 8. Multi-round active CMA-ES + projected AdamW refinement

Each topology is optimized in 18D continuous unit space. Three independently
seeded CMA trajectories run per topology. Later rounds warm-start two repeats from
same-topology/global incumbents and retain one cold repeat. CMA receives a
finite, non-overlapping feasibility tier, never a soft mixture that allows a
low objective to buy a hard-constraint violation.

Projected AdamW differentiates only the 21-input forward model's W/P/F terms.
CatBoost and L/D are not differentiable; every exact checkpoint is accepted
only after the authoritative evaluator confirms all hard gates and an improved
official score.


In [9]:
from bwb_pipeline.optimization import optimize_all_cases

optimization_result = optimize_all_cases(
    evaluator,
    missions,
    topology_prior,
    opt_config,
    seedbook,
    checkpoint_directory=OPTIMIZATION_ROOT,
    progress_callback=lambda row: print(
        f"case={row['case_id']} round={row['round']} "
        f"topology=({row['# of Ribs']},{row['# of Fuselage Ribs']},{row['# of Fuselage Spars']}) "
        f"repeat={row['repeat']} mode={row['start_mode']} "
        f"evals={row['actual_evaluations']}"
    ),
)
optimization_result.save(OPTIMIZATION_ROOT)


/Users/hmu2718/Downloads/bwb_reproducible_pipeline_2r_0.5/src/bwb_pipeline/models/forward.py:206: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/torch/csrc/utils/tensor_numpy.cpp:219.)
  batch = torch.as_tensor(


case=1 round=0 topology=(4,3,3) repeat=0 mode=cold_uniform evals=52960
case=1 round=0 topology=(4,3,3) repeat=1 mode=cold_uniform evals=52960
case=1 round=0 topology=(4,3,3) repeat=2 mode=cold_uniform evals=52960
case=1 round=0 topology=(5,3,3) repeat=0 mode=cold_uniform evals=14912
case=1 round=0 topology=(5,3,3) repeat=1 mode=cold_uniform evals=14912
case=1 round=0 topology=(5,3,3) repeat=2 mode=cold_uniform evals=14912
case=1 round=0 topology=(5,3,4) repeat=0 mode=cold_uniform evals=5248
case=1 round=0 topology=(5,3,4) repeat=1 mode=cold_uniform evals=5248
case=1 round=0 topology=(5,3,4) repeat=2 mode=cold_uniform evals=5248
case=1 round=0 topology=(5,5,3) repeat=0 mode=cold_uniform evals=1280
case=1 round=0 topology=(5,5,3) repeat=1 mode=cold_uniform evals=1280
case=1 round=0 topology=(5,5,3) repeat=2 mode=cold_uniform evals=1280
case=1 round=0 topology=(6,3,3) repeat=0 mode=cold_uniform evals=10080
case=1 round=0 topology=(6,3,3) repeat=1 mode=cold_uniform evals=10080
case=1 round